In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv("Perceptron_Table.csv")

display(df)

X = df.iloc[:, 0:3].values
d = df.iloc[:, -1].values

if df.iloc[:, -1].dtype == object:
    d = np.where(d == "P1", -1, 1)

d = d.astype(np.float32)
X = X.astype(np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X)
X = np.column_stack((-np.ones(len(X), dtype=np.float32), X))

In [ ]:
class StopOnConvergence(tf.keras.callbacks.Callback):
    def __init__(self, X_data, d_target):
        super().__init__()
        self.X_data = X_data
        self.d_target = d_target

    def on_epoch_end(self, epoch, logs=None):
        u = self.model(self.X_data, training=False).numpy().reshape(-1)
        y_pred = np.where(u >= 0, 1, -1)

        errors = np.sum(self.d_target != y_pred)

        if errors == 0:
            self.model.stop_training = True

results = []
accuracy_list = []

for i in range(1, 6):
    initializer = tf.keras.initializers.RandomUniform(minval=0., maxval=1., seed=i)

    inputs = tf.keras.Input(shape=(4,))
    outputs = tf.keras.layers.Dense(1, activation='linear', use_bias=False, kernel_initializer=initializer)(inputs)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    initial_weights = model.get_weights()[0].flatten()

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.05), loss=tf.keras.losses.Hinge())

    convergence_stop = StopOnConvergence(X, d)

    history = model.fit(X, d, epochs=5000, batch_size=len(X), verbose=0, callbacks=[convergence_stop])

    final_weights = model.get_weights()[0].flatten()
    num_epochs = len(history.history['loss'])

    u = model.predict(X, verbose=0).reshape(-1)
    y_pred = np.where(u >= 0, 1, -1)
    acc = accuracy_score(d, y_pred)
    accuracy_list.append(acc)

    results.append({
        'Training': f"T{i}",
        'w0_i': initial_weights[0],
        'w1_i': initial_weights[1],
        'w2_i': initial_weights[2],
        'w3_i': initial_weights[3],
        'w0_f': final_weights[0],
        'w1_f': final_weights[1],
        'w2_f': final_weights[2],
        'w3_f': final_weights[3],
        'Number of epochs': num_epochs
    })

df_results = pd.DataFrame(results)

multi_columns = pd.MultiIndex.from_tuples([
    ('', 'Training'),
    ('Initial weight vector', 'w0'),
    ('Initial weight vector', 'w1'),
    ('Initial weight vector', 'w2'),
    ('Initial weight vector', 'w3'),
    ('Final weight vector', 'w0'),
    ('Final weight vector', 'w1'),
    ('Final weight vector', 'w2'),
    ('Final weight vector', 'w3'),
    ('', 'Number of epochs')
])

df_results.columns = multi_columns
df_results = df_results.round(4)

display(df_results)

In [ ]:
for i, acc in enumerate(accuracy_list, start=1):
    print(f"Accuracy T{i}: {acc * 100:.2f}%")

mean_acc = np.mean(accuracy_list) * 100
print(f"\nMean accuracy: {mean_acc:.2f}%")

In [ ]:
# Example of automatic sample classification

data = [
    [-0.3665, 0.0620, 5.9891],
    [-0.7842, 1.1267, 5.5912],
    [0.3012, 0.5611, 5.8234],
    [0.7757, 1.0648, 8.0677],
    [0.1570, 0.8028, 6.3040],
    [-0.7014, 1.0316, 3.6005],
    [0.3748, 0.1536, 6.1537],
    [-0.6920, 0.9404, 4.4058],
    [-1.3970, 0.7141, 4.9263],
    [-1.8842, -0.2805, 1.2548]
]

df_samples = pd.DataFrame(data, columns=['x1', 'x2', 'x3'])
df_samples.index = np.arange(1, len(df_samples) + 1)
df_samples.index.name = 'Sample'

X_new = scaler.transform(df_samples.values)
X_new = np.column_stack((-np.ones(len(X_new), dtype=np.float32), X_new))

for i, res in enumerate(results, start=1):
    final_weights = np.array([res['w0_f'], res['w1_f'], res['w2_f'], res['w3_f']])

    u = np.dot(X_new, final_weights)
    y_pred = np.where(u >= 0, 1, -1)
    df_samples[f'y (T{i})'] = y_pred

display(df_samples)